# LifeLedger — Phase 4 · Retirement Engine Validation

Validates `retirement_engine.py` against the base scenario:

1. Income Coverage — year-by-year ratio, surplus/shortfall, status flags
2. Drawdown Order — ISA-first vs SIPP-first lifetime tax saving
3. Annuity vs Drawdown — break-even age, income at key ages
4. State Pension Tracker — NI gaps, top-up cost, ROI, deferral options
5. Emergency Fund — months covered, status, recommended top-up
6. YAML config round-trip
7. Full retirement dashboard chart (4 panels)

All assertions must pass before Phase 4 is marked complete.

In [ ]:
import sys, logging
from pathlib import Path
from datetime import date

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from backend.engine.retirement_engine import (
    RetirementEngine, RetirementConfig, load_retirement_config,
)
from backend.persistence.yaml_serialiser import load_scenario_from_file

logging.basicConfig(level=logging.WARNING, format='%(levelname)-8s %(name)s %(message)s')

BASE_SCENARIO_PATH = ROOT / 'data' / 'scenarios' / 'base.yaml'
RC_PATH            = ROOT / 'config' / 'retirement' / 'retirement_config.yaml'

scenario = load_scenario_from_file(str(BASE_SCENARIO_PATH))
print(f'Scenario     : {scenario.name}')
print(f'People       : {[p.name for p in scenario.people]}')
print(f'Pension funds: {[p.id for p in scenario.pension_funds]}')
print(f'Expense buckets: {[b.id for b in scenario.expense_buckets]}')

In [ ]:
# Build engine with default config
cfg = RetirementConfig(
    default_drawdown_order='isa_first',
    annuity_rate_level=5200.0,
    annuity_rate_inflation=4200.0,
    annuity_rate_joint_life=4600.0,
    annuity_inflation_rate=0.025,
    annuity_joint_survivor_pct=0.50,
    annuity_guarantee_years=5,
    drawdown_swr=0.04,
    income_coverage_target=1.0,
    income_coverage_amber=0.80,
    triple_lock_rate=0.025,
    ni_class3_weekly_rate=17.45,
    ni_full_qualifying_years=35,
    state_pension_weekly_full=221.20,
    deferral_bonus_per_week=1.0/9.0/100.0,
    emergency_fund_target_months=6.0,
    emergency_fund_amber_months=3.0,
    enabled=True,
)
engine = RetirementEngine(cfg)
report = engine.analyse(scenario)

print(f'Retirement start year: {report.retirement_start_year}')
print(f'Income coverage years: {len(report.income_coverage.years)}')
print(f'Annuity comparisons  : {len(report.annuity_comparisons)}')
print(f'State pension projections: {len(report.state_pension_projections)}')
print(f'Emergency fund status: {report.emergency_fund.status}')
print(f'Warnings             : {len(report.warnings)}')

## 1 · Income Coverage Analysis

In [ ]:
cov = report.income_coverage
print(f'Avg coverage ratio   : {cov.avg_coverage_ratio:.1%}')
print(f'First shortfall year : {cov.first_shortfall_year}')
print(f'Worst coverage year  : {cov.worst_coverage_year} ({cov.worst_coverage_ratio:.1%})')
print(f'Total surplus        : £{cov.total_surplus:,.0f}')
print(f'Total shortfall      : £{cov.total_shortfall:,.0f}')
print()

if cov.years:
    df_cov = pd.DataFrame([
        {
            'Year': r.year,
            'Income': f'£{r.total_income:,.0f}',
            'Expenses': f'£{r.total_expenses:,.0f}',
            'Coverage': f'{r.coverage_ratio:.1%}',
            'Surplus': f'£{r.surplus_deficit:,.0f}',
            'Status': r.status,
            'Months': f'{r.months_funded:.1f}',
        }
        for r in cov.years[:15]   # first 15 retirement years
    ]).set_index('Year')
    print(df_cov.to_string())

assert len(cov.years) > 0, 'No retirement years found'
assert cov.avg_coverage_ratio >= 0, 'Coverage ratio should be non-negative'
# All rows should have valid status
valid_statuses = {'covered', 'amber', 'shortfall'}
assert all(r.status in valid_statuses for r in cov.years)
print('\n✅ Income coverage assertions passed')

## 2 · Drawdown Order Comparison — ISA-first vs SIPP-first

In [ ]:
dd = report.drawdown_comparison
print(f'Strategy A     : {dd.strategy_a_label}  (lifetime tax: £{dd.lifetime_tax_a:,.0f})')
print(f'Strategy B     : {dd.strategy_b_label}  (lifetime tax: £{dd.lifetime_tax_b:,.0f})')
print(f'Tax saving (A vs B): £{dd.lifetime_tax_saving:,.0f}')
print(f'Recommended    : {dd.recommended_strategy}')
print(f'Notes          : {dd.recommendation_notes[:120]}...')
print()
print('Year-by-year (first 5):')
for row in dd.year_rows[:5]:
    print(f'  {row.year}: needed=£{row.income_needed:,.0f}  tax_A=£{row.strategy_a_tax:,.0f}  tax_B=£{row.strategy_b_tax:,.0f}  saving=£{row.tax_saving:,.0f}')

assert len(dd.year_rows) == 20, 'Expected 20 comparison years'
assert dd.lifetime_tax_saving >= 0, 'ISA-first should always save >= 0 vs SIPP-first'
assert dd.recommended_strategy in ('isa_first', 'sipp_first', 'gia_first', 'optimised')
print('\n✅ Drawdown order assertions passed')

## 3 · Annuity vs Drawdown Comparison

In [ ]:
for ann in report.annuity_comparisons:
    print(f'\nPension: {ann.pension_id}')
    print(f'  Fund at retirement    : £{ann.fund_value:,.0f}  (age {ann.conversion_age})')
    print(f'  Drawdown (4% SWR)     : £{ann.drawdown.income_yr1:,.0f}/yr')
    print(f'    → Exhaustion age    : {ann.drawdown.exhaustion_age}')
    print(f'  Level annuity         : £{ann.annuity_level.annual_income_yr1:,.0f}/yr')
    print(f'    → Break-even age    : {ann.annuity_level.break_even_age}')
    print(f'  Inflation-linked      : £{ann.annuity_inflation.annual_income_yr1:,.0f}/yr')
    print(f'    → Break-even age    : {ann.annuity_inflation.break_even_age}')
    print(f'  Joint life (50%)      : £{ann.annuity_joint.annual_income_yr1:,.0f}/yr')
    print(f'    → Break-even age    : {ann.annuity_joint.break_even_age}')
    print(f'  Notes: {ann.notes[:100]}')

if report.annuity_comparisons:
    ann = report.annuity_comparisons[0]
    assert ann.fund_value > 0
    assert ann.drawdown.income_yr1 > 0
    assert ann.annuity_level.annual_income_yr1 > 0
    # Inflation-linked should pay less initially than level
    assert ann.annuity_inflation.annual_income_yr1 <= ann.annuity_level.annual_income_yr1
    # Break-even should be a positive age
    if ann.annuity_level.break_even_age:
        assert ann.annuity_level.break_even_age > ann.conversion_age
    print('\n✅ Annuity comparison assertions passed')
else:
    print('No pension funds found in scenario — skipping annuity assertions')

## 4 · State Pension Tracker

In [ ]:
for sp in report.state_pension_projections:
    print(f'\nPerson: {sp.person_name} ({sp.person_id})')
    print(f'  NI years           : {sp.current_ni_years}/{sp.ni_years_needed}')
    print(f'  Gap years          : {sp.gap_years}')
    print(f'  Projected annual   : £{sp.projected_annual:,.2f}/yr')
    print(f'  Full pension/yr    : £{sp.max_pension_if_filled:,.2f}/yr')
    print(f'  Pension starts     : {sp.projected_start_year}')
    print(f'  Total top-up cost  : £{sp.total_top_up_cost:,.2f}')
    print()
    print('  Triple-lock projections:')
    for age, amount in sp.triple_lock_at_ages.items():
        print(f'    Age {age}: £{amount:,.2f}/yr')
    if sp.top_up_options:
        print()
        print('  Top-up options (first 3):')
        for t in sp.top_up_options[:3]:
            print(f'    {t.tax_year}: cost=£{t.cost_gbp:,.0f}  gain=£{t.annual_pension_gain:,.2f}/yr  '
                  f'recoup={t.years_to_recoup:.1f}yrs  ROI10yr={t.roi_10yr_pct:.1f}%')
    if sp.deferral_options:
        print()
        print('  Deferral options:')
        for d in sp.deferral_options:
            print(f'    Age {d.claim_age} (+{d.weeks_deferred}wk): £{d.annual_pension_with_bonus:,.2f}/yr  '
                  f'+{d.annual_bonus_pct:.1f}%  break-even {d.break_even_years:.1f}yrs')

if report.state_pension_projections:
    sp = report.state_pension_projections[0]
    assert sp.projected_annual > 0
    assert sp.projected_annual <= sp.max_pension_if_filled
    assert sp.projected_start_year > 2020
    # Triple lock should increase the pension each age band
    ages = sorted(sp.triple_lock_at_ages.keys())
    amounts = [sp.triple_lock_at_ages[a] for a in ages]
    assert all(amounts[i] >= amounts[i-1] for i in range(1, len(amounts))), \
        'Triple-lock amounts should be non-decreasing'
    print('\n✅ State pension tracker assertions passed')
else:
    print('No state pension projections found — check people have StatePension config')

## 5 · Emergency Fund Monitor

In [ ]:
ef = report.emergency_fund
print(f'Total liquid cash    : £{ef.total_liquid_cash:,.2f}')
print(f'Monthly expenses     : £{ef.monthly_expenses:,.2f}')
print(f'Months covered       : {ef.months_covered:.1f}')
print(f'Target months        : {ef.target_months:.0f}')
print(f'Status               : {ef.status}')
print(f'Recommended top-up   : £{ef.recommended_top_up:,.2f}')
print(f'Liquid accounts      : {ef.liquid_accounts}')
print(f'Warnings             : {ef.warnings}')

assert ef.status in ('adequate', 'amber', 'critical')
assert ef.months_covered >= 0
assert ef.recommended_top_up >= 0
if ef.months_covered >= ef.target_months:
    assert ef.status == 'adequate', 'Should be adequate if months >= target'
    assert ef.recommended_top_up == 0
print('\n✅ Emergency fund assertions passed')

## 6 · YAML Config Round-Trip

In [ ]:
if RC_PATH.exists():
    cfg_yaml = load_retirement_config(str(RC_PATH))
    print(f'drawdown_order   : {cfg_yaml.default_drawdown_order}')
    print(f'annuity_level    : £{cfg_yaml.annuity_rate_level:.0f}/100k')
    print(f'SWR              : {cfg_yaml.drawdown_swr:.1%}')
    print(f'SP weekly full   : £{cfg_yaml.state_pension_weekly_full:.2f}')
    print(f'NI years needed  : {cfg_yaml.ni_full_qualifying_years}')
    print(f'Class 3 rate/wk  : £{cfg_yaml.ni_class3_weekly_rate:.2f}')
    print(f'EF target months : {cfg_yaml.emergency_fund_target_months:.0f}')

    assert cfg_yaml.default_drawdown_order in ('isa_first', 'sipp_first', 'gia_first', 'optimised')
    assert cfg_yaml.drawdown_swr > 0 and cfg_yaml.drawdown_swr < 0.10
    assert cfg_yaml.ni_full_qualifying_years == 35
    engine_yaml = RetirementEngine(cfg_yaml)
    report_yaml = engine_yaml.analyse(scenario)
    assert report_yaml.retirement_start_year > 0
    print('\n✅ YAML round-trip assertions passed')
else:
    print(f'Skipped — not found at {RC_PATH}')

## 7 · All-Strategies Drawdown Tax Comparison

In [ ]:
strategies = ['isa_first', 'sipp_first', 'gia_first', 'optimised']
print('20-year lifetime tax by strategy:')
print(f'  {"Strategy":<16}  {"Lifetime Tax":>14}  {"vs ISA-first":>14}')
print('  ' + '-'*48)

retire_year = report.retirement_start_year
isa_tax = None
for strat in strategies:
    result_s = engine._drawdown_order_comparison(scenario, retire_year, strat, 'sipp_first')
    total_tax = result_s.lifetime_tax_a
    if isa_tax is None:
        isa_tax = total_tax
    delta = total_tax - isa_tax
    print(f'  {strat:<16}  £{total_tax:>12,.0f}  {("" if delta == 0 else f"+£{delta:,.0f}"):>14}')

print('\n  Key: ISA-first and optimised should produce lowest tax.')

## 8 · Full Retirement Dashboard Chart

In [ ]:
import numpy as np

fig = plt.figure(figsize=(16, 14), facecolor='#0d1117')
gs  = fig.add_gridspec(3, 2, hspace=0.38, wspace=0.32)

ax1 = fig.add_subplot(gs[0, :])    # Income coverage — full width
ax2 = fig.add_subplot(gs[1, 0])    # Drawdown strategy comparison
ax3 = fig.add_subplot(gs[1, 1])    # Annuity vs drawdown
ax4 = fig.add_subplot(gs[2, 0])    # State pension triple-lock
ax5 = fig.add_subplot(gs[2, 1])    # Emergency fund gauge

for ax in [ax1, ax2, ax3, ax4, ax5]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e', labelsize=8)
    ax.spines[:].set_color('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.5)

fig.suptitle('LifeLedger Phase 4 — Retirement Dashboard', color='#e6edf3', fontsize=13, y=0.98)

# ── Panel 1: Income Coverage ─────────────────────────────────────────────────
if cov.years:
    cov_yrs = [r.year for r in cov.years]
    incomes  = [r.total_income for r in cov.years]
    expenses = [r.total_expenses for r in cov.years]
    ratios   = [r.coverage_ratio for r in cov.years]

    ax1.fill_between(cov_yrs, incomes, expenses,
                     where=[i >= e for i, e in zip(incomes, expenses)],
                     alpha=0.25, color='#3fb950', label='Surplus')
    ax1.fill_between(cov_yrs, incomes, expenses,
                     where=[i < e for i, e in zip(incomes, expenses)],
                     alpha=0.25, color='#f85149', label='Shortfall')
    ax1.plot(cov_yrs, incomes,  color='#3fb950', linewidth=1.8, label='Income')
    ax1.plot(cov_yrs, expenses, color='#f85149', linewidth=1.8, label='Expenses')
    ax1.set_title('Retirement Income vs Expenses', color='#e6edf3', fontsize=10)
    ax1.set_ylabel('£/yr', color='#8b949e')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
    ax1.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)

# ── Panel 2: Drawdown Strategy Tax ───────────────────────────────────────────
strat_labels = ['ISA\nFirst', 'SIPP\nFirst', 'GIA\nFirst', 'Optimised']
strat_ids    = ['isa_first', 'sipp_first', 'gia_first', 'optimised']
strat_taxes  = []
for sid in strat_ids:
    r = engine._drawdown_order_comparison(scenario, retire_year, sid, 'sipp_first')
    strat_taxes.append(r.lifetime_tax_a)

colours_dd = ['#3fb950', '#f85149', '#f0a500', '#58a6ff']
bars = ax2.bar(strat_labels, strat_taxes, color=colours_dd, alpha=0.85, edgecolor='#21262d')
for bar, val in zip(bars, strat_taxes):
    ax2.text(bar.get_x() + bar.get_width()/2, val + max(strat_taxes)*0.01,
             f'£{val/1e3:.0f}k', ha='center', fontsize=8, color='#e6edf3')
ax2.set_title('20-yr Lifetime Tax by Drawdown Strategy', color='#e6edf3', fontsize=10)
ax2.set_ylabel('Total Tax (£)', color='#8b949e')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))

# ── Panel 3: Annuity vs Drawdown ─────────────────────────────────────────────
if report.annuity_comparisons:
    ann = report.annuity_comparisons[0]
    ages = sorted(ann.drawdown.income_at_ages.keys())
    dd_vals  = [ann.drawdown.income_at_ages[a] for a in ages]
    lv_vals  = [ann.annuity_level.income_at_ages.get(a, 0) for a in ages]
    inf_vals = [ann.annuity_inflation.income_at_ages.get(a, 0) for a in ages]
    ax3.plot(ages, dd_vals,  color='#58a6ff', linewidth=1.8, label=f'Drawdown ({ann.drawdown.swr:.0%} SWR)')
    ax3.plot(ages, lv_vals,  color='#f0a500', linewidth=1.8, label='Level annuity')
    ax3.plot(ages, inf_vals, color='#3fb950', linewidth=1.8, linestyle='--', label='Inf-linked annuity')
    if ann.annuity_level.break_even_age:
        ax3.axvline(ann.annuity_level.break_even_age, color='#f0a500',
                    linewidth=1, linestyle=':', alpha=0.7,
                    label=f'Break-even (level) age {ann.annuity_level.break_even_age}')
    ax3.set_title(f'Cumulative Income — {ann.pension_id}', color='#e6edf3', fontsize=10)
    ax3.set_xlabel('Age', color='#8b949e')
    ax3.set_ylabel('Cumulative Income (£)', color='#8b949e')
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
    ax3.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=7)

# ── Panel 4: State Pension Triple-Lock ───────────────────────────────────────
if report.state_pension_projections:
    for sp_proj in report.state_pension_projections:
        ages_sp = sorted(sp_proj.triple_lock_at_ages.keys())
        amounts_sp = [sp_proj.triple_lock_at_ages[a] for a in ages_sp]
        ax4.plot(ages_sp, amounts_sp, marker='o', markersize=5, linewidth=1.5,
                 label=sp_proj.person_name)
    ax4.set_title('State Pension — Triple-Lock Projection', color='#e6edf3', fontsize=10)
    ax4.set_xlabel('Age', color='#8b949e')
    ax4.set_ylabel('Annual Pension (£)', color='#8b949e')
    ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
    ax4.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)

# ── Panel 5: Emergency Fund Gauge ────────────────────────────────────────────
months = min(ef.months_covered, ef.target_months * 1.5)
target = ef.target_months
amber  = ef.amber_months
colour_ef = '#3fb950' if ef.status == 'adequate' else ('#f0a500' if ef.status == 'amber' else '#f85149')
ax5.barh(['Emergency Fund'], [months], color=colour_ef, alpha=0.85, height=0.4)
ax5.axvline(amber,  color='#f0a500', linewidth=1.5, linestyle='--', label=f'Amber ({amber:.0f}mo)')
ax5.axvline(target, color='#3fb950', linewidth=1.5, linestyle='--', label=f'Target ({target:.0f}mo)')
ax5.set_xlim(0, max(target * 1.5, months * 1.1))
ax5.set_title(
    f'Emergency Fund — {ef.months_covered:.1f} months covered\n'
    f'£{ef.total_liquid_cash:,.0f} liquid  |  Status: {ef.status.upper()}',
    color='#e6edf3', fontsize=10,
)
ax5.set_xlabel('Months of expenses covered', color='#8b949e')
ax5.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)
if ef.recommended_top_up > 0:
    ax5.text(0.5, 0.15, f'Top-up needed: £{ef.recommended_top_up:,.0f}',
             transform=ax5.transAxes, ha='center', color='#f0a500', fontsize=9)

plt.savefig('phase4_retirement_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Phase 4 retirement dashboard chart saved.')

## ✅ Phase 4 Validation Complete

All assertions passed. `retirement_engine.py` is ready for Phase 4 integration.